#----------PyTorch Tutorial 2-----------------

In [2]:
import torch
import numpy as np

In [ ]:
# %%
# empty, rand, zeros, ones
x = torch.ones(2, 3, dtype=torch.int)
x = torch.tensor([2.5, 2.93])
# x.dtype, x.size()
print(x)

In [ ]:
# Basic Arithmetic
x = torch.rand(2,2)
y = torch.rand(2,2)
# add, sub, mul, div
z = x+y
z = torch.add(x,y)
print(z)
print(y)
y.add_(x) # _ allows to perform the task in the vairable itself
print(y)

In [ ]:
# Slicing
x = torch.rand(4,4)
print(x)
print(x[:,0])
print(x[1,1].item()) # get a value directly
y = x.view(-1,8) # automatically calcualtes missing dimension
print(y)

In [ ]:
# Converting from numpy to torchtensor
a = torch.ones(5)
print(a)
b = a.numpy() # conversion
print(b)
# Note: Be careful: if tensor on Cpu and not gpu, then both objects share same memory location, 
# if we change one, the other changes too. For example:
a.add_(1)
print(a)
print(b) # ones were added to b as well, as they point to the same memory location

# Converting from torchtensor to numpy
a = np.ones(5)
print(a)
b = torch.from_numpy(a) # conversion
print(b)
a += 1
print(a)
print(b) # similarly tensor gets modified too

In [ ]:
# How to perform operations with CUDA, creating a tensor in GPU
if torch.cuda.is_available():
    device = torch.device("cuda")
    x = torch.ones(5, device=device) # directly creating in GPU
    y = torch.ones(5) # creating on CPU
    y = y.to(device) # converting to GPU tensor
    z = x + y
    # z.numpy() #-> Will call an error, numpy only handles cpu tensors
    z = z.to("cpu") # converts to CPU tensor
    print(z)

#--------PyTorch Tutorial 3 [Gradient Calculation with Autograd]-------------

In [ ]:
# Autograd package to caluclate gradients for model optimization
# Note: 
# for Gaussian (bell curve), theoretically ranges btwn -inf to inf, but in practice 99.7% of values fall btwn -3 and 3
# for Uniform, returns value [0,1), every number within range has an equal probability

# x = torch.rand(3) # Samples from uniform distribution

# Step 1: Create x. requires_grad=True tells PyTorch "I want to calculate gradients with respect to this tensor later". 
# This is how you flag which variables you want to learn/optimize.
x = torch.randn(3, requires_grad=True) # Samples from a Standard Normal Distribution (Gaussian distribution)
print(x)

# Step 2: Forward pass (Computing values)
# PyTorch is silently building a graph behind the scenes tracking every operation. 
y = x+2
print(y)

# Only Y:
#  ---Forward-->
#       ___
# x -->| + |------> y ------->|
# 2 -->|___|                  | [gradient Function]
#  <--Add Backward (dy/dx)<---|
# With graph and backpropogation we get gradient

print(x.grad) # Output: None,...No gradient? Only forward pass occurred
z = y*y*2
print(z)
z = z.mean()
print(z)
# Complete Diagram
# x --> [+2] --> y --> [*y*2] --> z --> [.mean()] --> z(scalar)

# Step 3: Backward Pass (Computing gradients)
z.backward() # PyTorch walks the graph BACKWARDS computing gradients
print(x.grad) # dz/dx <-- Gradient exists after calling .backward(). 
              # Before that, PyTorch has only done the forward pass (computing values), not the backward pass (computing gradients).

tensor([ 2.2657, -0.3437, -0.9012], requires_grad=True)
tensor([4.2657, 1.6563, 1.0988], grad_fn=<AddBackward0>)
None
tensor([36.3918,  5.4867,  2.4149], grad_fn=<MulBackward0>)
tensor(14.7645, grad_fn=<MeanBackward0>)
tensor([5.6876, 2.2084, 1.4651])


What is a Gradient?  
A gradient is simply how much z changes when x changes slightly — mathematically it's a derivative (dz/dx). This is the core of how neural networks learn.  

We define:  
z = mean(2(x+2)²)  
z = (1/3) * 2(x+2)²        # mean over 3 elements -> A scalar result of forward pass  
z.backward                 # Vector-Jacobian Product  
dz/dx = (4/3)(x+2)         # derivative [chain rule] -> x.grad  

Example: x.grad[0] = 5.69, if we increase x[0] by a tiny amount E, then z goes up by 5.69*E, its a rate of change, not value itself.  

Why this matter:  
In a neural network:  

x = the weights of the network  
z = the loss (how wrong the network is)  
dz/dx = which direction to nudge the weights to reduce the loss  

That's training in a nutshell — compute loss, call .backward(), use gradients to update weights. Repeat thousands of times.  

How VJP works:  
Remember: We are going backwards now:  
x = [2.2657, -0.3437, -0.9012]  
y = [4.2657,  1.6563,  1.0988]   (y = x + 2)  
z_vec = [36.39, 5.49, 2.41]      (z_vec = 2y²)  
z = 14.7645                       (z = mean(z_vec))  

J1 = mean layer (z_vec -> z), shape 1x3  
J2 = multiply layer (y -> z_vec), shape 3x3  
J3 = dd layer (x -> y), shape 3x3  
v, v1, v2, v3 = upstream gradient, starts at 1 (scalar loss)  

After J1: v1 = v.J1 = [1] · [0.333, 0.333, 0.333] = [0.333, 0.333, 0.333]  
After J2: v2 = v1.J2 = [0.333, 0.333, 0.333] · diag(17.063, 6.625, 4.395) = [5.6876,  2.2084,  1.4651]  
After J3: x.grad = v₃ = v₂ · J₃ = [5.6876, 2.2084, 1.4651] · I = [5.6876, 2.2084, 1.4651]  

In [3]:
# Before we would pass z as scalar (using mean), where v = 1 , but what about using it as a vector?
x = torch.randn(3, requires_grad=True)
print(x)
y = x+2
print(y)
z = y*y*2
# z = z.mean() # converts to scalar
print(z) # currently z is a vector

v = torch.tensor([0.1,1.0,0.001], dtype=torch.float32) # provides v manually as a starting gradient
# Above we are using weighing gradients of each output differently [1-> we care alot about z[1], 0.001 -> we almost ignore z[2]] 
z.backward(v)
print(x.grad)

tensor([ 0.0585, -1.3057,  0.5222], requires_grad=True)
tensor([2.0585, 0.6943, 2.5222], grad_fn=<AddBackward0>)
tensor([ 8.4751,  0.9641, 12.7226], grad_fn=<MulBackward0>)
tensor([0.8234, 2.7772, 0.0101])


In [ ]:
# 3 ways to stop creating gradient functions/tracking history in computational graph 
x = torch.randn(3, requires_grad=True)
print(x)
# Option 1
x.requires_grad_(False)
print(x)
# Option 2
y = x.detach()
print(y)
# Option 3
with torch.no_grad():
    y = x+2
    print(y)

tensor([-0.2252, -0.4669, -0.4102], requires_grad=True)
tensor([-0.2252, -0.4669, -0.4102])
tensor([-0.2252, -0.4669, -0.4102])
tensor([1.7748, 1.5331, 1.5898])


In [ ]:
# Training a model
# weights = torch.ones(4, requires_grad=True)

# for add in range(3):
#     output = (weights*3).sum()
#     output.backward()
#     print(weights.grad)
#     weights.grad.zero_() #Prevents accumulating of gradients

# #Optimization
# optimizer = torch.optim.SGD([weights], lr = 0.01) # SGD: Stochastic Gradient Descent
# optimizer.step()
# optimizer.zero_grad()
# print(optimizer)

weights = torch.ones(4, requires_grad=True)

optimizer = torch.optim.SGD([weights], lr=0.01)  # list

for epoch in range(3):
    output = (weights * 3).sum()
    
    output.backward()
    print(weights.grad)
    
    # Optimizer formula: wt_new = wt_old - lr*grad

    optimizer.step()        # update weights using grad
    print(weights.data)     # .data to clear requires_grad=True clutter in output
    # optimizer.zero_grad()   # clear grad for next iteration
    # weights.grad.zero_()  # no longer needed, optimizer handles this


tensor([3., 3., 3., 3.])
tensor([0.9700, 0.9700, 0.9700, 0.9700])
tensor([6., 6., 6., 6.])
tensor([0.9100, 0.9100, 0.9100, 0.9100])
tensor([9., 9., 9., 9.])
tensor([0.8200, 0.8200, 0.8200, 0.8200])


#----------PyTorch Tutorial 4 [Backpropagation]-----------------

x (input) -> [a(x)] -> y(output, input) -> [b(y)] -> z(output)
              ^dy/dx                         ^dz/dy
Chain Rule: dz/dx = dz/dy . dy/dx

Computational Graph:
For each operation, Pytorch does this in background:
x--|f = x.y|-> z -> ..... -> Loss function (dloss/dx = ?)
y--|  [*]  |
Local gradients: dz/dx = dx.y/dx = y , dz/dy = dx.y/dy = x

Applying Chain rule to get gradient of loss: dLoss/dx = dLoss/dz . dz/dx

So,
1) Forward pass: Compute Loss
2) Compute local gradients
3) Backward pass: Compute dLoss/dWeights using the Chain Rule

Taking Linear Regression as an example:
y_pred = w.x
loss = (y_pred-y_actual)^2 = (wx-y_actual)^2

x ->| * | -> y_pred -> |-| -> s -> [^2] -> Loss
w ->|   |  y_actual -> | |

Minimize loss: Finding derivative of Loss wrt weight

1st step -> Forward pass: x*y -> y_pred-y_actual -> ^2 -> loss
2nd step -> Local gradients: dloss/ds, ds/dy_pred , dy_pred/dw
3rd step -> Backward pass: using chain rule: dloss/dw = dloss/ds . ds/dy_pred . dy_pred/dw

Example:
x = 1, y = 2, w = 1
1st step: y* = 1.1 = 1 | s = 1-2 = -1 | loss = (-1)^2 = 1
2nd step: dloss/ds = ds^2/ds = 2s | ds/dy* = dy*-y/dy* = 1 | dy*/dw = dwx/dw = x
3rd step: dloss/dy* = dloss/ds . ds/dy* = 2s*1 = -2 | dloss/dw = dloss/dy* . dy*/dw = -2.x = -2

In [ ]:
x = torch.tensor(1.0)
y = torch.tensor(2.0)
w = torch.tensor(1.0, requires_grad=True)

#forward pass and compute loss:
y_hat = w*x
loss = (y_hat - y)**2
print(loss)

#backward pass:
loss.backward()
print(w.grad)
### update weights
### next forward and backward


tensor(1., grad_fn=<PowBackward0>)
tensor(-2.)
tensor(1., requires_grad=True)
